# Praktikum Pengolahan Citra: Brightness, Histogram Equalization, dan Boundary Filter
Citra: *Mandrill/Baboon* (512×512, RGB). Jalankan sel dari atas ke bawah. Pada Colab, unggah `baboon24.jpg` saat diminta.

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt
from google.colab import files
up = files.upload()                       # pilih baboon24.jpg
bgr = cv2.imread(list(up.keys())[0])
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
g   = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
print(bgr.shape, 'mean %.1f std %.1f min %d max %d' % (g.mean(), g.std(), g.min(), g.max()))
print('persentil 1/5/50/95/99:', np.percentile(g,[1,5,50,95,99]))

## 1. Identifikasi citra

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(15,4.5))
ax[0].imshow(rgb); ax[0].set_title('Citra asli (RGB)')
ax[1].imshow(g,cmap='gray',vmin=0,vmax=255); ax[1].set_title('Grayscale')
ax[2].hist(g.ravel(),bins=256,range=(0,256),color='gray'); ax[2].set_title('Histogram citra asli')
ax[2].set_xlabel('Intensitas'); ax[2].set_ylabel('Jumlah piksel')
ax[0].axis('off'); ax[1].axis('off'); plt.tight_layout(); plt.show()

**Masalah yang terlihat:**
- Kontras sedang-rendah pada area wajah: hidung dan batang hidung berada pada rentang abu-abu yang sempit (±100–140) sehingga tampak datar.
- Histogram tidak merata: dua puncak (±125 dan ±180), nyaris kosong di bawah 30 dan di atas 205. Rentang dinamis hanya terpakai ±0–231 dengan 99% piksel berada di 32–203.
- Piksel menyimpang: bulu/kumis yang tipis dan terang di atas latar gelap menghasilkan piksel yang sangat berbeda dari tetangganya (tekstur kasar, mirip derau).
- Kecerahan keseluruhan sudah cukup (rata-rata ±130), jadi tidak terlalu gelap/terang.

## 2. Brightness adjustment

In [ ]:
fig,ax=plt.subplots(2,3,figsize=(15,9)); bright={}
for i,b in enumerate([-50,0,50]):
    r=np.clip(g.astype(np.int16)+b,0,255).astype(np.uint8); bright[b]=r
    ax[0,i].imshow(r,cmap='gray',vmin=0,vmax=255); ax[0,i].set_title(f'beta = {b:+d}'); ax[0,i].axis('off')
    ax[1,i].hist(r.ravel(),bins=256,range=(0,256),color='gray'); ax[1,i].set_xlim(0,255)
    print(f'beta {b:+d}: mean {r.mean():.1f} std {r.std():.1f} piksel=0 {(r==0).mean()*100:.2f}% piksel=255 {(r==255).mean()*100:.2f}%')
plt.tight_layout(); plt.show()

**Analisis:** beta hanya menggeser histogram; standar deviasi (kontras) tetap ±42. beta=-50 membuat 3,96% piksel terpotong menjadi 0 (detail area gelap hilang); beta=+50 membuat 0,75% piksel jenuh di 255 (bulu putih memudar). Detail tidak menjadi lebih mudah dilihat: yang berubah hanya kecerahan, bukan pemisahan antar-objek. beta=0 tetap yang terbaik di antara ketiganya.

## 3. Histogram equalization

In [ ]:
eq=cv2.equalizeHist(g)
fig,ax=plt.subplots(2,2,figsize=(12,10))
ax[0,0].imshow(g,cmap='gray',vmin=0,vmax=255); ax[0,0].set_title('Sebelum')
ax[0,1].imshow(eq,cmap='gray',vmin=0,vmax=255); ax[0,1].set_title('Sesudah (equalization)')
ax[1,0].hist(g.ravel(),bins=256,range=(0,256),color='gray'); ax[1,0].set_title('Histogram sebelum')
ax[1,1].hist(eq.ravel(),bins=256,range=(0,256),color='gray'); ax[1,1].set_title('Histogram sesudah')
ax[0,0].axis('off'); ax[0,1].axis('off'); plt.tight_layout(); plt.show()
def ent(x):
    p=np.bincount(x.ravel(),minlength=256)/x.size; p=p[p>0]; return -(p*np.log2(p)).sum()
print('std %.1f -> %.1f | entropi %.3f -> %.3f | level unik %d -> %d' % (g.std(),eq.std(),ent(g),ent(eq),len(np.unique(g)),len(np.unique(eq))))

**Analisis:** std naik dari 42,5 menjadi 73,7 dan rentang intensitas melebar ke 0–255. Garis kerut pipi biru, kulit hidung, dan area gelap di sekitar mata menjadi jauh lebih terpisah. Histogram menjadi lebih rata (tidak sempurna karena citra diskret) dan tampak bergerigi/ada celah; jumlah level unik turun 231 → 161 sehingga entropi turun sedikit (7,37 → 7,25). Efek samping: tekstur bulu dan derau ikut menguat sehingga citra terlihat lebih kasar.

## 4. Boundary filter (8 piksel tetangga)
Nilai piksel pusat dibandingkan dengan rata-rata 8 tetangganya. Dua varian: **(A)** semua piksel diganti rata-rata 8 tetangga; **(B)** bersyarat: hanya piksel dengan |p − rata-rata8| > T yang diganti.

In [ ]:
k=np.ones((3,3),np.float32); k[1,1]=0; k/=8
gf=g.astype(np.float32)
mean8=cv2.filter2D(gf,-1,k,borderType=cv2.BORDER_REFLECT)
bfA=np.clip(mean8,0,255).astype(np.uint8)          # varian A
T=40; dev=np.abs(gf-mean8); mask=dev>T
bfB=np.where(mask,bfA,g).astype(np.uint8)          # varian B
print(f'Piksel menyimpang (T={T}): {mask.mean()*100:.2f}%')
for t in (20,30,40,60): print(f'  T={t}: {(dev>t).mean()*100:.2f}% piksel diganti')
lapvar=lambda x: cv2.Laplacian(x,cv2.CV_64F).var()
def sob(x): return np.hypot(cv2.Sobel(x,cv2.CV_64F,1,0),cv2.Sobel(x,cv2.CV_64F,0,1)).mean()
mse=lambda a,b: np.mean((a.astype(float)-b.astype(float))**2)
for n,x in [('asli',g),('boundary A (penuh)',bfA),('boundary B (bersyarat)',bfB),('equalization',eq)]:
    print(f'{n:24s} varLaplacian {lapvar(x):9.1f}  Sobel {sob(x):6.2f}  MSE vs asli {mse(x,g):8.2f}')
fig,ax=plt.subplots(2,3,figsize=(15,10))
for a,x,t in zip(ax[0],[g,bfA,bfB],['Asli','Boundary A (semua piksel)',f'Boundary B (T={T})']):
    a.imshow(x,cmap='gray',vmin=0,vmax=255); a.set_title(t); a.axis('off')
ax[1,0].imshow(mask,cmap='gray'); ax[1,0].set_title('Piksel menyimpang terdeteksi')
ax[1,1].imshow(cv2.absdiff(g,bfA),cmap='gray'); ax[1,1].set_title('|asli - A|')
ax[1,2].imshow(cv2.absdiff(g,bfB),cmap='gray'); ax[1,2].set_title('|asli - B|')
for a in ax[1]: a.axis('off')
plt.tight_layout(); plt.show()

**Analisis:** varian A menghaluskan seluruh citra (varians Laplacian 4830 → 352, rata-rata gradien Sobel 117 → 64): piksel menyimpang hilang, tetapi bulu, kumis, dan garis pipi ikut kabur. Varian B hanya mengganti 6,8% piksel, sehingga derau berkurang namun detail utama relatif terjaga (varians Laplacian 2891). Catatan: pada citra ini "piksel menyimpang" sebagian besar adalah tekstur bulu asli, bukan derau, sehingga filtering menghilangkan informasi nyata.

## 5. Kesimpulan

| Teknik | Hasil pada citra ini | Penilaian |
|---|---|---|
| Brightness ±50 | Kontras tidak berubah; ada piksel terpotong (0 / 255) | Tidak sesuai |
| Histogram equalization | std 42,5 → 73,7; detail pipi, hidung, mata lebih jelas | **Paling sesuai** |
| Boundary filter | Derau berkurang, tetapi detail dan ketajaman turun | Sesuai hanya sebagai langkah opsional (varian B) |

**Teknik paling sesuai: histogram equalization**, karena masalah utama citra ini adalah kontras sedang dan distribusi intensitas yang tidak merata (dua puncak, rentang 32–203), dan itu yang diperbaiki langsung oleh teknik ini. Brightness tidak menyentuh masalah tersebut, sedangkan boundary filter mengatasi masalah yang bukan masalah utama dan mengorbankan tekstur. Jika ingin hasil lebih bersih, dapat dipertimbangkan boundary filter bersyarat dahulu, lalu equalization (atau CLAHE untuk membatasi penguatan derau).